In [5]:
# Load the raw orderlines file into a dataframe we can clean.
import pandas as pd
orderlines = pd.DataFrame(pd.read_csv('orderlines.csv'))
df_orderlines = orderlines.copy()

In [6]:
# Look at the raw data.
df_orderlines

,id,id_order,product_id,product_quantity,sku,unit_price,date
0,1119109,299539,0,1,OTT0133,18.99,2017-01-01 00:07:19
1,1119110,299540,0,1,LGE0043,399.00,2017-01-01 00:19:45
2,1119111,299541,0,1,PAR0071,474.05,2017-01-01 00:20:57
3,1119112,299542,0,1,WDT0315,68.39,2017-01-01 00:51:40
4,1119113,299543,0,1,JBL0104,23.74,2017-01-01 01:06:38
...,...,...,...,...,...,...,...
293978,1650199,527398,0,1,JBL0122,42.99,2018-03-14 13:57:25
293979,1650200,527399,0,1,PAC0653,141.58,2018-03-14 13:57:34
293980,1650201,527400,0,2,APP0698,9.99,2018-03-14 13:57:41
293981,1650202,527388,0,1,BEZ0204,19.99,2018-03-14 13:58:01


In [7]:
# Check the shape and column types.
df_orderlines.info()
df_orderlines.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id                293983 non-null  int64 
 1   id_order          293983 non-null  int64 
 2   product_id        293983 non-null  int64 
 3   product_quantity  293983 non-null  int64 
 4   sku               293983 non-null  object
 5   unit_price        293983 non-null  object
 6   date              293983 non-null  object
dtypes: int64(4), object(3)
memory usage: 15.7+ MB


(293983, 7)

In [8]:
# Check for fully duplicate rows.
df_orderlines.duplicated().sum()


np.int64(0)

In [9]:
# Check if any 'id' values repeat (they shouldn't -- it's the row key).
df_orderlines['id'].duplicated().sum()

np.int64(0)

1. Change type of unit_price to float64

In [10]:
# Check for missing values in each column.
df_orderlines.isna().sum()

id                  0
id_order            0
product_id          0
product_quantity    0
sku                 0
unit_price          0
date                0
dtype: int64

In [11]:
# Drop the product_id column -- we don't need it.
df_orderlines = df_orderlines.drop(columns = 'product_id')

In [12]:
# Check the columns again after dropping product_id.
df_orderlines.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id                293983 non-null  int64 
 1   id_order          293983 non-null  int64 
 2   product_quantity  293983 non-null  int64 
 3   sku               293983 non-null  object
 4   unit_price        293983 non-null  object
 5   date              293983 non-null  object
dtypes: int64(3), object(3)
memory usage: 13.5+ MB


In [13]:
# Look at summary stats for the numeric columns.
df_orderlines.describe()

,id,id_order,product_quantity
count,2.939830e+05,293983.000000,293983.000000
mean,1.397918e+06,419999.116544,1.121126
std,1.530096e+05,66344.486479,3.396569
min,1.119109e+06,241319.000000,1.000000
25%,1.262542e+06,362258.500000,1.000000
50%,1.406940e+06,425956.000000,1.000000
75%,1.531322e+06,478657.000000,1.000000
max,1.650203e+06,527401.000000,999.000000


In [14]:
# (Tried converting unit_price to numbers directly -- didn't work, left as a note.)
 #pd.to_numeric(df_orderlines['unit_price'])


In [15]:
# See what happens if we force unit_price to numbers -- bad values become NaN.
numeric_attempt = pd.to_numeric(df_orderlines['unit_price'], errors='coerce')
numeric_attempt

0          18.99
1         399.00
2         474.05
3          68.39
4          23.74
           ...  
293978     42.99
293979    141.58
293980      9.99
293981     19.99
293982     13.99
Name: unit_price, Length: 293983, dtype: float64

In [16]:
# Pull out the unit_price values that failed to convert, and the orders they belong to.
failed = df_orderlines['unit_price'][numeric_attempt.isna() & df_orderlines['unit_price'].notna()]
id_order_mask = df_orderlines.loc[numeric_attempt.isna(), 'id_order']


In [17]:
# (An earlier idea for fixing the price format -- not used, kept as a note.)
#df_orderlines_copy['unit_price'] = df_orderlines_copy['unit_price'].str.replace(r'\.(?=.*\.)', '', regex=True).astype('float64')

In [18]:
# Drop every line belonging to an order that had a bad unit_price.
df_orderlines = df_orderlines.loc[~df_orderlines.id_order.isin(id_order_mask)]


In [19]:
# Check how many decimal points are left in unit_price now.
df_orderlines.unit_price.str.count(r'\.').value_counts()

unit_price
1    216250
Name: count, dtype: int64

In [20]:
# Convert unit_price to a proper number.
df_orderlines['unit_price'] = pd.to_numeric(df_orderlines['unit_price'], errors='coerce')

In [21]:
# Check the columns again now that unit_price is numeric.
df_orderlines.info()

<class 'pandas.core.frame.DataFrame'>
Index: 216250 entries, 0 to 293982
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                216250 non-null  int64  
 1   id_order          216250 non-null  int64  
 2   product_quantity  216250 non-null  int64  
 3   sku               216250 non-null  object 
 4   unit_price        216250 non-null  float64
 5   date              216250 non-null  object 
dtypes: float64(1), int64(3), object(2)
memory usage: 11.5+ MB


In [22]:
# Check for missing values again.
df_orderlines.isna().sum()

id                  0
id_order            0
product_quantity    0
sku                 0
unit_price          0
date                0
dtype: int64

In [23]:
# Drop any remaining rows with missing values.
df_orderlines = df_orderlines.dropna(axis=0)

In [24]:
# Check the columns once more.
df_orderlines.info()

<class 'pandas.core.frame.DataFrame'>
Index: 216250 entries, 0 to 293982
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                216250 non-null  int64  
 1   id_order          216250 non-null  int64  
 2   product_quantity  216250 non-null  int64  
 3   sku               216250 non-null  object 
 4   unit_price        216250 non-null  float64
 5   date              216250 non-null  object 
dtypes: float64(1), int64(3), object(2)
memory usage: 11.5+ MB


In [25]:
# Look at the smallest unit_price values and count how many are negative.
df_orderlines.nsmallest(10, 'unit_price')
df_orderlines[df_orderlines['unit_price'] < 0].count()

id                  1
id_order            1
product_quantity    1
sku                 1
unit_price          1
date                1
dtype: int64

In [26]:
# Remove the one row with a negative price.
df_orderlines = df_orderlines[df_orderlines['unit_price'] >= 0]

In [27]:
# Check the columns after removing it.
df_orderlines.info()

<class 'pandas.core.frame.DataFrame'>
Index: 216249 entries, 0 to 293982
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                216249 non-null  int64  
 1   id_order          216249 non-null  int64  
 2   product_quantity  216249 non-null  int64  
 3   sku               216249 non-null  object 
 4   unit_price        216249 non-null  float64
 5   date              216249 non-null  object 
dtypes: float64(1), int64(3), object(2)
memory usage: 11.5+ MB


Date parsing  

In [28]:
# Convert the date column from text to a real date.
df_orderlines['date'] = pd.to_datetime(df_orderlines['date'], errors = 'coerce')

In [29]:
# Check the columns after fixing the date.
df_orderlines.info()

<class 'pandas.core.frame.DataFrame'>
Index: 216249 entries, 0 to 293982
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   id                216249 non-null  int64         
 1   id_order          216249 non-null  int64         
 2   product_quantity  216249 non-null  int64         
 3   sku               216249 non-null  object        
 4   unit_price        216249 non-null  float64       
 5   date              216249 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(3), object(1)
memory usage: 11.5+ MB


In [30]:
# Confirm no dates failed to convert.
df_orderlines['date'].isna().sum()

np.int64(0)

In [31]:
# List the remaining columns.
df_orderlines.columns

Index(['id', 'id_order', 'product_quantity', 'sku', 'unit_price', 'date'], dtype='object')

In [32]:
# Count how many lines each order has.
df_orderlines['id_order'].value_counts()

id_order
428186    32
406616    17
323269    17
332942    15
319417    15
          ..
299558     1
527401     1
299560     1
299561     1
295310     1
Name: count, Length: 170213, dtype: int64

In [33]:
# Check for duplicate rows one more time.
df_orderlines.duplicated().sum()

np.int64(0)

In [34]:
# See how many rows we ended up with.
df_orderlines.shape[0]

216249

In [35]:
# Save the cleaned orderlines to a new CSV file.
df_orderlines.to_csv('orderlines.clean.csv')
